In [1]:
#!/usr/bin/env python3
"""
================================================================================
HazardNet: Production Multi-API Data Pipeline, GEE Exporter & Deduplicator
================================================================================
Author: Ashif Ahmed Shuvo (BAU Agrometeorology)
Description: Complete pipeline for fetching multi-hazard disaster records across
             Bangladesh's 64 districts, generating dense daily spatio-temporal
             grids, building GEE export tensors, running analytics, and performing
             temporal/spatial deduplication.
================================================================================
"""

import os
import re
import xml.etree.ElementTree as ET
import zipfile
import logging
from datetime import datetime, timedelta
from io import BytesIO
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ------------------------------------------------------------------------------
# LOGGING & GLOBAL CONFIGURATION
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("HazardNet")

CONFIG: Dict[str, Any] = {
    "APP_NAME": "BAU-HazardNetResearch-AAS7016",
    "RW_BASE_URL": "https://api.reliefweb.int/v2",
    "GDACS_RSS_URL": "https://www.gdacs.org/xml/rss.xml",
    "IFRC_API_BASE": "https://goadmin.ifrc.org/api/v2/",
    "EONET_URL": "https://eonet.gsfc.nasa.gov/api/v3/events",
    "HDX_RESOURCE_ID": "c5ce40d6-07b1-4f36-955a-d6196436ff6b",
    "COUNTRY_CODE": "BD",
    "GEE_WINDOW_DAYS": 50,
    "OUTPUT_FILE": "HazardNet_Master_Dataset_Final.csv",
    "DISTRICT_COORDS": {
        'Barguna': [22.1509, 90.1263], 'Barishal': [22.7010, 90.3535], 'Bhola': [22.6859, 90.6483],
        'Jhalokati': [22.6406, 90.1989], 'Patuakhali': [22.3596, 90.3297], 'Pirojpur': [22.5791, 89.9751],
        'Bandarban': [22.1953, 92.2184], 'Brahmanbaria': [23.9571, 91.1119], 'Chandpur': [23.2321, 90.6631],
        'Chattogram': [22.3569, 91.7832], 'Cumilla': [23.4607, 91.1809], "Cox's Bazar": [21.4272, 92.0058],
        'Feni': [23.0159, 91.3976], 'Khagrachari': [23.1192, 91.9847], 'Lakshmipur': [22.9447, 90.8282],
        'Noakhali': [22.8695, 91.0994], 'Rangamati': [22.6574, 92.1733], 'Dhaka': [23.8103, 90.4125],
        'Faridpur': [23.6071, 89.8429], 'Gazipur': [24.0023, 90.4264], 'Gopalganj': [23.0051, 89.8267],
        'Kishoreganj': [24.4260, 90.9821], 'Madaripur': [23.1641, 90.1833], 'Manikganj': [23.8644, 89.9967],
        'Munshiganj': [23.5435, 90.5354], 'Narayanganj': [23.6238, 90.5000], 'Narsingdi': [23.9322, 90.7154],
        'Rajbari': [23.7574, 89.6444], 'Shariatpur': [23.2423, 90.4348], 'Tangail': [24.2513, 89.9167],
        'Bagerhat': [22.6516, 89.7859], 'Chuadanga': [23.6401, 88.8504], 'Jessore': [23.1664, 89.2081],
        'Jhenaidah': [23.5450, 89.1726], 'Khulna': [22.8456, 89.5403], 'Kushtia': [23.9013, 89.1204],
        'Magura': [23.4873, 89.4199], 'Meherpur': [23.7622, 88.6318], 'Narail': [23.1725, 89.5126],
        'Satkhira': [22.7185, 89.0705], 'Jamalpur': [24.9375, 89.9311], 'Mymensingh': [24.7471, 90.4031],
        'Netrokona': [24.8700, 90.7275], 'Sherpur': [25.0205, 90.0154], 'Bogra': [24.8481, 89.3730],
        'Joypurhat': [25.0947, 89.0209], 'Naogaon': [24.7936, 88.9318], 'Natore': [24.4102, 88.9805],
        'Chapainawabganj': [24.5965, 88.2742], 'Pabna': [24.0063, 89.2493], 'Rajshahi': [24.3745, 88.6042],
        'Sirajganj': [24.4577, 89.7080], 'Dinajpur': [25.6217, 88.6354], 'Gaibandha': [25.3288, 89.5280],
        'Kurigram': [25.8054, 89.6361], 'Lalmonirhat': [25.9129, 89.4426], 'Nilphamari': [25.9317, 88.8560],
        'Panchagarh': [26.3411, 88.5541], 'Rangpur': [25.7439, 89.2752], 'Thakurgaon': [26.0337, 88.4616],
        'Habiganj': [24.3749, 91.4133], 'Moulvibazar': [24.4829, 91.7606], 'Sunamganj': [25.0658, 91.3950],
        'Sylhet': [24.8949, 91.8687]
    }
}

BD_DISTRICTS: List[str] = list(CONFIG['DISTRICT_COORDS'].keys())

REGIONAL_SYNONYMS: Dict[str, List[str]] = {
    "North Bengal": ["Dinajpur", "Gaibandha", "Kurigram", "Lalmonirhat", "Nilphamari", "Panchagarh", "Rangpur", "Thakurgaon"],
    "Coastal Belt": ["Barguna", "Bhola", "Patuakhali", "Satkhira", "Khulna", "Bagerhat"],
    "Barind Tract": ["Rajshahi", "Naogaon", "Chapainawabganj"]
}

HAZARD_MAPPER: Dict[str, int] = {
    'Flood': 0, 'Flash Flood': 0, 'Tropical Cyclone': 1, 'Drought': 2,
    'Heat Wave': 3, 'Cold Wave': 4, 'Salinity': 5, 'Saltwater Intrusion': 5,
    'Epidemic': 6, 'Fire': 7, 'Earthquake': 8, 'Landslide': 9, 'Severe Local Storm': 10
}

SEVERITY_INDEX_MAP: Dict[str, str] = {
    'Red': 'CRITICAL_RISK',
    'Orange': 'MAJOR_RISK',
    'Yellow': 'MODERATE_RISK'
}

CURRENT_YEAR: int = datetime.now().year


# ------------------------------------------------------------------------------
# 1. SPATIAL & DATA AGGREGATION UTILITIES
# ------------------------------------------------------------------------------
def robust_spatial_ner(text: str) -> str:
    """Extract affected Bangladesh districts from text using NER matching and regional rules."""
    if not text:
        return "National"
    if "national" in text.lower():
        return "National"
    
    found = [d for d in BD_DISTRICTS if d.lower() in text.lower()]
    for region, districts in REGIONAL_SYNONYMS.items():
        if region.lower() in text.lower():
            found.extend(districts)
            
    common_fixes = ["Sylhet", "Chattogram", "Cox's Bazar", "Rajshahi", "Khulna"]
    for d in common_fixes:
        if d.lower() in text.lower():
            found.append(d)
            
    unique_found = list(set(found))
    return "|".join(unique_found) if unique_found else "National"


def fetch_emdat_from_hdx() -> Optional[pd.DataFrame]:
    """Fetch official EM-DAT disaster dataset for Bangladesh from HDX API."""
    try:
        url = f"https://data.humdata.org/api/3/action/resource_show?id={CONFIG['HDX_RESOURCE_ID']}"
        res = requests.get(url, timeout=15).json()
        file_url = res['result']['url']
        file_data = requests.get(file_url, timeout=30).content
        
        df = pd.read_excel(BytesIO(file_data))
        rename_logic = {
            'Glide': 'GLIDE', 'GLIDE': 'GLIDE', 'Disaster No.': 'GLIDE', 'Dis No': 'GLIDE',
            'Total Deaths': 'Deaths', 'Deaths': 'Deaths', 'Total affected': 'Affected',
            'Total Affected': 'Affected', 'Total damage (': 'Damage_USD'
        }
        
        new_cols = {}
        for col in df.columns:
            for key, val in rename_logic.items():
                if key in col:
                    new_cols[col] = val
        df.rename(columns=new_cols, inplace=True)
        return df
    except Exception as e:
        logger.error(f"EM-DAT Fetch Error: {e}")
        return None


def get_eonet_data() -> List[Dict[str, Any]]:
    """Fetch active global natural events from NASA EONET v3 API."""
    try:
        res = requests.get(CONFIG["EONET_URL"], params={"status": "all", "days": 11000}, timeout=20)
        return res.json().get('events', [])
    except Exception as e:
        logger.warning(f"EONET API request failed: {e}")
        return []


def match_eonet_proximity(date_str: str, districts: str, eonet_list: List[Dict[str, Any]]) -> int:
    """Calculate proximity matches between event record and NASA EONET observations."""
    matches = 0
    try:
        if not isinstance(date_str, str) or not isinstance(districts, str):
            return 0
        row_date = datetime.strptime(date_str, '%Y-%m-%d')
        district_list = districts.split('|')
        
        for e in eonet_list:
            geoms = e.get('geometry', [])
            if not isinstance(geoms, list) or not geoms:
                continue
            e_date_str = geoms[0].get('date', '')[:10]
            if not e_date_str:
                continue
            try:
                e_date = datetime.strptime(e_date_str, '%Y-%m-%d')
            except ValueError:
                continue
                
            if abs((row_date - e_date).days) <= 5:
                e_title = e.get('title', '')
                if any(d.lower() in e_title.lower() for d in district_list):
                    matches += 1
    except Exception as e:
        logger.debug(f"EONET match error: {e}")
        return 0
    return matches


def get_crop_data_from_reports(glide_id: str) -> str:
    """Query ReliefWeb reports for specific GLIDE ID to retrieve impact text."""
    if not glide_id or glide_id == "N/A":
        return ""
    try:
        res = requests.post(
            f"{CONFIG['RW_BASE_URL']}/reports",
            json={
                "filter": {"field": "glide", "value": [glide_id]},
                "fields": {"include": ["body"]},
                "limit": 5
            },
            timeout=10
        ).json()
        return " ".join([item['fields'].get('body', '') for item in res.get('data', [])])
    except Exception as e:
        logger.debug(f"ReliefWeb crop report query failed for {glide_id}: {e}")
        return ""


def enrich_all_sources(df: pd.DataFrame, eonet_events: List[Dict[str, Any]]) -> pd.DataFrame:
    """Enrich core events with IFRC GO, GDACS, DesInventar, and EONET metadata."""
    if df.empty:
        return df

    # IFRC GO API
    try:
        ifrc = requests.get(
            f"{CONFIG['IFRC_API_BASE']}event/?countries__iso3={CONFIG['COUNTRY_CODE']}&limit=200",
            timeout=15
        ).json().get('results', [])
        ifrc_map = {e.get('glide'): e for e in ifrc if e.get('glide')}
        df['IFRC_Severity'] = df['GLIDE'].map(lambda x: ifrc_map.get(x, {}).get('ifrc_severity_level_display'))
        df['IFRC_Affected'] = df['GLIDE'].map(lambda x: ifrc_map.get(x, {}).get('num_affected', 0))
    except Exception as e:
        logger.warning(f"IFRC enrichment failed: {e}")

    # GDACS RSS feed
    try:
        root = ET.fromstring(requests.get(CONFIG['GDACS_RSS_URL'], timeout=15).content)
        gdacs_glides = []
        for item in root.findall('.//item'):
            title = item.find('title').text if item.find('title') is not None else ''
            match = re.search(r'GLIDE:\s*([A-Z0-9\-]+)', title)
            if match:
                gdacs_glides.append(match.group(1))
        df['GDACS_Active_Alert'] = df['GLIDE'].apply(lambda x: 1 if x in gdacs_glides else 0)
    except Exception as e:
        logger.warning(f"GDACS RSS enrichment failed: {e}")

    # DesInventar database
    try:
        resp = requests.get("https://www.desinventar.net/DesInventar/download/DI_export_bd.zip", timeout=20)
        with zipfile.ZipFile(BytesIO(resp.content)) as z:
            with z.open("DI_export_bd.xml") as f:
                di_map = {}
                for event in ET.parse(f).getroot().findall('.//fichas/TR'):
                    gl = event.find('glide').text if event.find('glide') is not None else None
                    if gl:
                        crops = sum([
                            float(i.find('valor').text) for i in event.findall('impactos/IMPACTO')
                            if 'crop' in (i.find('tipo').text or '').lower()
                        ])
                        di_map[gl] = crops
                df['DesInventar_Crop_Loss_Ha'] = df['GLIDE'].map(di_map)
    except Exception as e:
        logger.warning(f"DesInventar enrichment failed: {e}")

    # EONET spatial proximity
    df['EONET_Proximity_Matches'] = df.apply(
        lambda r: match_eonet_proximity(r['Date'], r['Location_Districts'], eonet_events),
        axis=1
    )
    return df


def run_hazardnet_unified_pipeline(start_date: str = "2000-01-01T00:00:00+00:00", end_date: Optional[str] = None) -> pd.DataFrame:
    """Main aggregator fetching ReliefWeb, EM-DAT, and satellite APIs into single DataFrame."""
    logger.info("Launching Unified HazardNet Aggregator...")
    if end_date is None:
        end_date = datetime.now().strftime('%Y-%m-%dT%H:%M:%S+00:00')

    rw_res = requests.post(
        f"{CONFIG['RW_BASE_URL']}/disasters?appname={CONFIG['APP_NAME']}",
        json={
            "filter": {"operator": "AND", "conditions": [
                {"field": "primary_country", "value": ["Bangladesh"]},
                {"field": "date.event", "value": {"from": start_date, "to": end_date}}
            ]},
            "fields": {"include": ["id", "glide", "name", "date.event", "type", "profile", "description"]},
            "limit": 1000,
            "sort": ["date.event:desc"]
        },
        timeout=25
    ).json().get('data', [])

    emdat_df = fetch_emdat_from_hdx()
    eonet_events = get_eonet_data()

    ml_records = []
    for entry in rw_res:
        f = entry['fields']
        eid, glide = entry['id'], f.get('glide', 'N/A')
        date_str = f.get('date', {}).get('event', '2000-01-01')[:10]
        dt = datetime.strptime(date_str, '%Y-%m-%d')
        desc = f.get('description', '')
        locs = robust_spatial_ner(desc)
        hazard_raw = f.get('type', [{}])[0].get('name', 'Unknown')
        
        rep_txt = get_crop_data_from_reports(glide)
        full_description = (desc + " " + rep_txt).strip() or "No detailed description available."
        
        crop_matches = re.findall(r'([\d,.]+)\s*(?:ha|hectares|acres)', (desc + " " + rep_txt).lower())
        total_ha = sum(float(c.replace(',', '')) for c in crop_matches if c.replace(',', '').replace('.', '').isdigit())
        
        ml_records.append({
            "Event_ID": eid, "GLIDE": glide, "Date": date_str, "Year": dt.year,
            "Hazard_Type": hazard_raw, "Hazard_Class": HAZARD_MAPPER.get(hazard_raw, 10),
            "Location_Districts": locs, "RW_Affected": f.get('profile', {}).get('affected', 0),
            "Text_Extracted_Crop_Ha": total_ha, "Full_Description": full_description,
            "GEE_Start": (dt - timedelta(days=CONFIG['GEE_WINDOW_DAYS'])).strftime('%Y-%m-%d'),
            "GEE_End": (dt + timedelta(days=CONFIG['GEE_WINDOW_DAYS'])).strftime('%Y-%m-%d')
        })

    df = pd.DataFrame(ml_records)
    if emdat_df is not None and 'GLIDE' in emdat_df.columns:
        merge_cols = [c for c in ['GLIDE', 'Deaths', 'Affected', 'Damage_USD'] if c in emdat_df.columns]
        df = pd.merge(df, emdat_df[merge_cols].dropna(subset=['GLIDE']), on='GLIDE', how='left')

    df = enrich_all_sources(df, eonet_events)
    affected_cols = [c for c in ['RW_Affected', 'IFRC_Affected', 'Affected'] if c in df.columns]
    df['Validated_Affected'] = df[affected_cols].max(axis=1).fillna(0)
    df['Last_Updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    df.to_csv(CONFIG['OUTPUT_FILE'], index=False)
    logger.info(f"Successfully aggregated {len(df)} raw events into {CONFIG['OUTPUT_FILE']}")
    return df


# ------------------------------------------------------------------------------
# 2. PRODUCTION PIPELINE CLASS (Spatio-Temporal Grid & GEE Formatting)
# ------------------------------------------------------------------------------
class HazardNetPipeline:
    def __init__(self, start_year: int = 2000, end_year: int = CURRENT_YEAR,
                 district_coords: Optional[Dict[str, List[float]]] = None,
                 severity_index_map: Optional[Dict[str, str]] = None):
        self.start_date = f"{start_year}-01-01"
        self.end_date = datetime.now().strftime("%Y-%m-%d")
        self.master_df: Optional[pd.DataFrame] = None
        self.localized_df: Optional[pd.DataFrame] = None
        self.dense_df: Optional[pd.DataFrame] = None
        self.district_coords = district_coords or CONFIG['DISTRICT_COORDS']
        self.severity_index_map = severity_index_map or SEVERITY_INDEX_MAP

    def step1_fetch_and_aggregate(self) -> None:
        logger.info("[Step 1] Fetching raw multi-source data...")
        self.master_df = run_hazardnet_unified_pipeline(
            start_date=f"{pd.to_datetime(self.start_date).year}-01-01T00:00:00+00:00",
            end_date=f"{pd.to_datetime(self.end_date).year}-12-31T23:59:59+00:00"
        )
        # Expand "National" events to all 64 districts
        national_rows = self.master_df['Location_Districts'] == 'National'
        if national_rows.any():
            all_districts_str = "|".join(BD_DISTRICTS)
            self.master_df.loc[national_rows, 'Location_Districts'] = all_districts_str

    def step2_localize_and_geocode(self) -> None:
        logger.info("[Step 2] Exploding locations to District-level geodata...")
        if self.master_df is None:
            raise ValueError("Master DataFrame is empty. Run Step 1 first.")
            
        df = self.master_df.copy()
        df['District'] = df['Location_Districts'].str.split('|')
        df = df.explode('District')
        df = df[df['District'].isin(self.district_coords.keys())]
        df = df.drop_duplicates(subset=['Date', 'District'], keep='first')
        
        df['Latitude'] = df['District'].map(lambda d: self.district_coords[d][0])
        df['Longitude'] = df['District'].map(lambda d: self.district_coords[d][1])
        self.localized_df = df

    def step3_apply_severity_indexing(self) -> None:
        logger.info("[Step 3] Applying HazardNet Severity Indexing Rules...")
        if self.localized_df is None:
            raise ValueError("Localized DataFrame is empty. Run Step 2 first.")

        def get_sev(row: pd.Series) -> str:
            ifrc = row.get('IFRC_Severity')
            if ifrc in ['Red', 'Orange', 'Yellow']:
                return ifrc
            return 'Red' if row.get('GDACS_Active_Alert') == 1 else 'Yellow'

        self.localized_df['API_Severity_Level'] = self.localized_df.apply(get_sev, axis=1)
        self.localized_df['Severity_Index_Name'] = self.localized_df['API_Severity_Level'].map(self.severity_index_map)
        self.localized_df['Severity_Score'] = self.localized_df['API_Severity_Level'].map({'Red': 3, 'Orange': 2, 'Yellow': 1})

    def step4_generate_dense_history(self) -> pd.DataFrame:
        logger.info(f"[Step 4] Generating dense {self.start_date[:4]}-{self.end_date[:4]} temporal grid...")
        if self.localized_df is None:
            raise ValueError("Localized DataFrame missing. Run Step 3 first.")

        dates = pd.date_range(start=self.start_date, end=self.end_date)
        districts = list(self.district_coords.keys())
        
        multi_idx = pd.MultiIndex.from_product([dates, districts], names=['Date', 'District'])
        grid = pd.DataFrame(index=multi_idx).reset_index()
        grid['Date'] = grid['Date'].dt.strftime('%Y-%m-%d')

        final = pd.merge(grid, self.localized_df, on=['Date', 'District'], how='left')
        fill_cols = {
            'Hazard_Type': 'Baseline/None',
            'Severity_Index_Name': 'NO_RISK',
            'Severity_Score': 0,
            'Validated_Affected': 0,
            'Full_Description': ''
        }
        for col, val in fill_cols.items():
            final[col] = final[col].fillna(val)

        final['Latitude'] = final['District'].map(lambda d: self.district_coords[d][0])
        final['Longitude'] = final['District'].map(lambda d: self.district_coords[d][1])

        self.dense_df = final
        output_file = 'HazardNet_Final_Production_Dataset.csv'
        self.dense_df.to_csv(output_file, index=False)
        logger.info(f"Pipeline Complete. Production Dataset saved to {output_file} ({len(self.dense_df)} rows).")
        return self.dense_df

    def run_full_pipeline(self) -> pd.DataFrame:
        self.step1_fetch_and_aggregate()
        self.step2_localize_and_geocode()
        self.step3_apply_severity_indexing()
        return self.step4_generate_dense_history()


def prepare_gee_dataset(df: pd.DataFrame, start_yr: int = 2000, end_yr: int = CURRENT_YEAR, window: int = 50) -> pd.DataFrame:
    """Extract non-baseline hazard events formatted with GEE temporal bounds for satellite downloading."""
    logger.info(f"Extracting GEE events ({start_yr}-{end_yr}) with ±{window}-day temporal windows...")
    df_prep = df.copy()
    df_prep['Date'] = pd.to_datetime(df_prep['Date'])
    
    df_prep = df_prep[(df_prep['Date'].dt.year >= start_yr) & (df_prep['Date'].dt.year <= end_yr)]
    df_events = df_prep[df_prep['Hazard_Type'] != 'Baseline/None'].copy().reset_index(drop=True)

    df_events['Event_ID_Internal'] = [f"Event_{i:04d}" for i in range(len(df_events))]
    df_events['GEE_Start'] = (df_events['Date'] - timedelta(days=window)).dt.strftime('%Y-%m-%d')
    df_events['GEE_End'] = (df_events['Date'] + timedelta(days=window)).dt.strftime('%Y-%m-%d')
    df_events['Date'] = df_events['Date'].dt.strftime('%Y-%m-%d')

    output_cols = [
        'Event_ID_Internal', 'GLIDE', 'Date', 'District', 'Latitude', 'Longitude',
        'Hazard_Type', 'GEE_Start', 'GEE_End', 'Severity_Index_Name', 'Severity_Score', 'Validated_Affected'
    ]
    if 'Full_Description' in df_events.columns:
        output_cols.append('Full_Description')

    return df_events[output_cols]


# ------------------------------------------------------------------------------
# 3. STATISTICAL ANALYSIS & EXPORTS
# ------------------------------------------------------------------------------
def run_all_analysis(df_events: pd.DataFrame, df_dense: pd.DataFrame) -> None:
    """Generate statistical summary reports and save structured CSV analytical tables."""
    logger.info("Executing Statistical Analysis & CSV Exports...")

    # 1. General Summary
    general_stats = pd.DataFrame({
        'Metric': ['Total Unique Events', 'Unique Districts', 'Date Range Start', 'Date Range End', 'Mean Severity Score', 'Total Validated Affected'],
        'Value': [len(df_events), df_events['District'].nunique(), df_events['Date'].min(), df_events['Date'].max(), df_events['Severity_Score'].mean(), df_events['Validated_Affected'].sum()]
    })
    general_stats.to_csv('hazardnet_general_summary_stats.csv', index=False)

    # 2. Hazard Type Breakdown
    hazard_stats = df_events.groupby('Hazard_Type').agg({
        'Event_ID_Internal': 'count',
        'Severity_Score': ['mean', 'std', 'max'],
        'Validated_Affected': ['sum', 'mean', 'max']
    }).reset_index()
    hazard_stats.columns = ['Hazard_Type', 'Event_Count', 'Mean_Severity', 'Std_Severity', 'Max_Severity', 'Total_Affected', 'Mean_Affected', 'Max_Affected']
    hazard_stats.to_csv('hazardnet_hazard_type_analysis.csv', index=False)

    # 3. District Vulnerability Index
    district_stats = df_events.groupby('District').agg({
        'Hazard_Type': 'nunique',
        'Event_ID_Internal': 'count',
        'Severity_Score': 'mean',
        'Validated_Affected': 'sum'
    }).reset_index()
    district_stats.columns = ['District', 'Unique_Hazard_Types', 'Total_Events', 'Avg_Severity', 'Cumulative_Affected']
    district_stats['Vulnerability_Score'] = (district_stats['Total_Events'] * district_stats['Avg_Severity']).rank(pct=True)
    district_stats.sort_values('Vulnerability_Score', ascending=False, inplace=True)
    district_stats.to_csv('hazardnet_district_vulnerability_index.csv', index=False)

    # 4. Correlation Matrix
    corr_matrix = df_events[['Severity_Score', 'Validated_Affected', 'Latitude', 'Longitude']].corr()
    corr_matrix.to_csv('hazardnet_statistical_correlations.csv')

    # 5. Yearly Temporal Trends
    yearly_trends = df_events.groupby(pd.to_datetime(df_events['Date']).dt.year).agg({
        'Event_ID_Internal': 'count',
        'Validated_Affected': 'sum'
    }).reset_index()
    yearly_trends.columns = ['Year', 'Event_Frequency', 'Annual_Affected_Population']
    yearly_trends.to_csv('hazardnet_yearly_temporal_trends.csv', index=False)

    logger.info("All statistical analytical reports exported to CSV.")


# ------------------------------------------------------------------------------
# 4. VISUALIZATION GENERATOR
# ------------------------------------------------------------------------------
def generate_all_visualizations(df_events: pd.DataFrame, df_dense: pd.DataFrame) -> None:
    """Generate static Seaborn/Matplotlib and interactive Plotly visualization figures."""
    logger.info("Generating hazard figures and maps...")
    
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['savefig.dpi'] = 300
    sns.set_theme(style='whitegrid')

    # 1. 25-Year Trend
    df_events_plot = df_events.copy()
    df_events_plot['Year'] = pd.to_datetime(df_events_plot['Date']).dt.year
    yearly_counts = df_events_plot.groupby('Year').size()
    
    plt.figure(figsize=(14, 6))
    sns.lineplot(x=yearly_counts.index, y=yearly_counts.values, marker='o', color='#d62728', linewidth=2.5)
    plt.title('25-Year Hazard Frequency Trend in Bangladesh (2000-2026)', fontsize=14)
    plt.xlabel('Year')
    plt.ylabel('Number of Events')
    plt.tight_layout()
    plt.savefig('hazard_frequency_trend_2000_2026.png')
    plt.close()

    # 2. Hazard Distribution
    plt.figure(figsize=(12, 7))
    hazard_order = df_events_plot['Hazard_Type'].value_counts().index
    sns.countplot(data=df_events_plot, y='Hazard_Type', order=hazard_order, palette='viridis', hue='Hazard_Type', legend=False)
    plt.title('Distribution of Hazard Types (Total Occurrences)')
    plt.tight_layout()
    plt.savefig('hazard_type_distribution.png')
    plt.close()

    # 3. Interactive Maps
    district_risk = df_dense.groupby('District').agg({
        'Severity_Score': 'mean',
        'Latitude': 'first',
        'Longitude': 'first'
    }).reset_index()
    
    fig = px.scatter_mapbox(
        district_risk, lat='Latitude', lon='Longitude', color='Severity_Score',
        size='Severity_Score', hover_name='District', zoom=6,
        mapbox_style='carto-positron', title='Mean Hazard Severity Index by District',
        color_continuous_scale=px.colors.sequential.OrRd
    )
    fig.write_html('district_risk_map.html')
    logger.info("Visualizations and interactive HTML map generated.")


# ------------------------------------------------------------------------------
# 5. HAZARDNET COMPREHENSIVE DEDUPLICATION ENGINE
# ------------------------------------------------------------------------------
class HazardNetDeduplicator:
    """Detects exact, near (±7-day window), temporal, and date duplicates."""

    def __init__(self, excluded_hazards: Optional[List[str]] = None):
        self.excluded_hazards = excluded_hazards or ['Epidemic']

    def load_and_prepare(self, csv_path: str) -> pd.DataFrame:
        logger.info(f"Loading dataset for deduplication check: {csv_path}")
        df = pd.read_csv(csv_path)
        df['Date'] = pd.to_datetime(df['Date'])

        orig_count = len(df)
        df_filtered = df[~df['Hazard_Type'].isin(self.excluded_hazards)].copy().reset_index(drop=True)
        
        removed = orig_count - len(df_filtered)
        logger.info(f"Excluded {self.excluded_hazards}. Remaining rows: {len(df_filtered)} (Removed {removed}).")
        return df_filtered

    def check_exact_duplicates(self, df: pd.DataFrame) -> int:
        exact_dups = df.duplicated(subset=['Date', 'Hazard_Type', 'District'], keep=False)
        count = int(exact_dups.sum())
        logger.info(f"Check 1 - Exact Duplicates (Same Date, Hazard, District): {count}")
        return count

    def check_near_duplicates_7days(self, df: pd.DataFrame) -> List[Dict[str, Any]]:
        logger.info("Check 2 - Near Duplicates (±7 Days, Same Hazard, Same District)...")
        duplications = []
        visited_indices = set()

        for idx, row in df.iterrows():
            if idx in visited_indices:
                continue

            date = row['Date']
            district = row['District']
            hazard = row['Hazard_Type']

            date_min = date - timedelta(days=7)
            date_max = date + timedelta(days=7)

            mask = (
                (df['Date'] >= date_min) &
                (df['Date'] <= date_max) &
                (df['District'] == district) &
                (df['Hazard_Type'] == hazard)
            )
            sim_indices = df[mask].index.tolist()

            if len(sim_indices) > 1:
                visited_indices.update(sim_indices)
                duplications.append({
                    'district': district,
                    'hazard_type': hazard,
                    'count': len(sim_indices),
                    'dates': df.loc[sim_indices, 'Date'].tolist(),
                    'indices': sim_indices
                })

        logger.info(f"Found {len(duplications)} near-duplicate groups within ±7-day window.")
        return duplications

    def check_temporal_clusters(self, df: pd.DataFrame) -> List[Dict[str, Any]]:
        clusters = []
        for district in df['District'].unique():
            dist_df = df[df['District'] == district]
            for hazard in dist_df['Hazard_Type'].unique():
                events = dist_df[dist_df['Hazard_Type'] == hazard]
                if len(events) > 1:
                    clusters.append({
                        'district': district,
                        'hazard_type': hazard,
                        'count': len(events),
                        'year_range': f"{events['Date'].min().year} - {events['Date'].max().year}"
                    })
        logger.info(f"Check 3 - Temporal Clusters across districts: {len(clusters)}")
        return clusters

    def check_date_clusters(self, df: pd.DataFrame) -> List[Dict[str, Any]]:
        """Completed function: find dates with multiple hazard records in the same district."""
        logger.info("Check 4 - Date Clusters (Multiple Events in Same District on Same Date)...")
        clusters = []
        grouped = df.groupby(['Date', 'District'])
        
        for (date, district), group in grouped:
            if len(group) > 1:
                clusters.append({
                    'date': date,
                    'district': district,
                    'count': len(group),
                    'hazards': group['Hazard_Type'].tolist()
                })
                
        logger.info(f"Found {len(clusters)} date-district multi-hazard cluster instances.")
        return clusters

    def deduplicate_and_save(self, df: pd.DataFrame, duplications: List[Dict[str, Any]], output_path: str) -> pd.DataFrame:
        """Remove near duplicates (keeping first record in each window) and save clean CSV."""
        indices_to_remove = set()
        for dup in duplications:
            sorted_indices = sorted(dup['indices'])
            indices_to_remove.update(sorted_indices[1:])

        df_cleaned = df.drop(index=list(indices_to_remove)).reset_index(drop=True)
        df_cleaned['Date'] = df_cleaned['Date'].dt.strftime('%Y-%m-%d')
        df_cleaned.to_csv(output_path, index=False)

        logger.info(f"Deduplication Complete: Removed {len(indices_to_remove)} redundant records.")
        logger.info(f"Cleaned dataset saved to: {output_path} ({len(df_cleaned)} rows)")
        return df_cleaned


# ------------------------------------------------------------------------------
# MAIN EXECUTION ENTRYPOINT
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    logger.info("==================================================================")
    logger.info("HAZARDNET PRODUCTION PIPELINE & DEDUPLICATION RUNNER")
    logger.info("==================================================================")

    # 1. Run Data Aggregation & Spatio-Temporal Pipeline
    pipeline = HazardNetPipeline(start_year=2000, end_year=CURRENT_YEAR)
    dense_grid_df = pipeline.run_full_pipeline()

    # 2. Extract GEE Tensors
    gee_df = prepare_gee_dataset(dense_grid_df, start_yr=2000, end_yr=CURRENT_YEAR, window=CONFIG['GEE_WINDOW_DAYS'])
    gee_file = 'HazardNet_Events_For_GEE.csv'
    gee_df.to_csv(gee_file, index=False)
    logger.info(f"GEE events dataset saved to {gee_file}.")

    # 3. Statistical Analysis & Visualizations
    run_all_analysis(gee_df, dense_grid_df)
    generate_all_visualizations(gee_df, dense_grid_df)

    # 4. Deduplication Engine Run
    dedup_engine = HazardNetDeduplicator(excluded_hazards=['Epidemic'])
    if Path(gee_file).exists():
        raw_events = dedup_engine.load_and_prepare(gee_file)
        dedup_engine.check_exact_duplicates(raw_events)
        near_dups = dedup_engine.check_near_duplicates_7days(raw_events)
        dedup_engine.check_temporal_clusters(raw_events)
        dedup_engine.check_date_clusters(raw_events)

        clean_output = 'BGD_climatic_hazards_dataset_2000_2026.csv'
        dedup_engine.deduplicate_and_save(raw_events, near_dups, clean_output)

    logger.info("==================================================================")
    logger.info("ALL PIPELINE STAGES COMPLETED SUCCESSFULLY")
    logger.info("==================================================================")

[2026-09-22 18:34:45] INFO - ==================================================================
[2026-09-22 18:34:45] INFO - HAZARDNET PRODUCTION PIPELINE & DEDUPLICATION RUNNER
[2026-09-22 18:34:45] INFO - ==================================================================
[2026-09-22 18:34:45] INFO - [Step 1] Fetching raw multi-source data...
[2026-09-22 18:34:45] INFO - Launching Unified HazardNet Aggregator...
[2026-09-22 18:35:12] WARNING - EONET API request failed: HTTPSConnectionPool(host='eonet.gsfc.nasa.gov', port=443): Read timed out. (read timeout=20)
[2026-09-22 18:36:04] WARNING - DesInventar enrichment failed: File is not a zip file
[2026-09-22 18:36:04] INFO - Successfully aggregated 70 raw events into HazardNet_Master_Dataset_Final.csv
[2026-09-22 18:36:04] INFO - [Step 2] Exploding locations to District-level geodata...
[2026-09-22 18:36:04] INFO - [Step 3] Applying HazardNet Severity Indexing Rules...
[2026-09-22 18:36:04] INFO - [Step 4] Generating dense 2000-2026 tem